### 🖼️ Dataset Thumbnails (hackelle___BigEarthNetV2-Lithuania-Summer-LMDB)

![Thumbnail](../thumbnails/hackelle___BigEarthNetV2-Lithuania-Summer-LMDB_01.png)

![Thumbnail](../thumbnails/hackelle___BigEarthNetV2-Lithuania-Summer-LMDB_02.png)

![Thumbnail](../thumbnails/hackelle___BigEarthNetV2-Lithuania-Summer-LMDB_03.png)

In [ ]:
import numpy as np
import random
from datasets import load_dataset, get_dataset_config_names
import pprint

# ========================================================================
# 🚀 데이터셋 정보 분석 (Tutor's Corner!)
# 🎨 데이터셋명: hackelle/BigEarthNetV2-Lithuania-Summer-LMDB
# 🌎 주제: 리투아니아 여름 지역의 위성 이미지 분류 (Remote Sensing Image Classification)
# 💡 데이터의 의미: 이 데이터셋은 여러 위성(Sentinel-1, Sentinel-2)에서 얻은
#    지구 표면의 작은 패치(Patch) 이미지를 모아놓은 것입니다.
#    우리가 해야 할 일은 이 이미지 패치가 무엇을 보여주는지 (예: 숲, 도시, 농경지)
#    를 '클래스 레이블'을 통해 분류하는 것입니다. 즉, '어떤 곳인지'를 AI에게 알려주는 과정입니다.
# 🛠️ 핵심 기능: 단순 이미지 분류를 넘어, '날씨 조건(구름, 눈)' 같은 복잡한 메타데이터를
#    분류 모델에 결합하여 성능을 높이는 것이 이 과제의 핵심 목표입니다!
# ========================================================================

# --- 설정 변수 ---
DATASET_ID = "hackelle/BigEarthNetV2-Lithuania-Summer-LMDB"
TARGET_SPLIT = "all_data"  # 사용할 스플릿 지정
SAMPLE_COUNT = 5  # 분석에 사용할 샘플 개수 (너무 많으면 느려요!)

print("=" * 80)
print("💡 초보자용 AI 실습: 위성 이미지 메타데이터 분석 튜토리얼을 시작합니다!")
print("=" * 80)

# 1. 사용 가능한 Config 목록 확인 (필수 과정!)
try:
    configs = get_dataset_config_names(DATASET_ID)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 기본 Config가 없거나 하나만 있는 경우를 대비
    if configs:
        selected_config = configs[0]
    else:
        selected_config = None
        
except Exception as e:
    print(f"ℹ️ Config 조회 중 오류 발생: {e}")
    selected_config = None

# 2. 데이터 로딩 시도 (스트리밍 vs. 일반 모드)
dataset = None
sample_data_list = []

print("\n🤖 데이터 로드를 시도합니다. 스트리밍 모드로 빠르게 진행해 볼까요?")

try:
    # Step 2-1: 스트리밍 로드 시도 (가장 빠르고 효율적입니다!)
    dataset = load_dataset(DATASET_ID, split=TARGET_SPLIT, streaming=True)
    print("🎉 스트리밍 모드(streaming=True)로 성공적으로 연결되었습니다! (메모리 효율 최고!)")

except Exception as e:
    # 스트리밍이 복잡하거나 실패할 경우, 일반 모드로 제한된 데이터를 다운로드합니다.
    print(f"⚠️ 스트리밍 모드 로드 실패 또는 복잡합니다. ({e}). 일반 모드로 {SAMPLE_COUNT}개만 다운로드하여 진행할게요.")
    try:
        dataset = load_dataset(DATASET_ID, split=TARGET_SPLIT, streaming=False)
    except Exception as e_fallback:
        print(f"🚨 필수 라이브러리나 인터넷 연결을 확인해주세요. 데이터셋 로드에 실패했습니다: {e_fallback}")
        exit()

# 3. 데이터 샘플링 및 이터레이터 설정 (제한된 크기만 사용)
print("\n✨ 상위 몇 개 샘플만 골라와서 분석할 거예요. 전체 데이터를 다 쓰면 컴퓨터가 힘들어해요!")

# 제약사항 준수: len() 사용 금지, .take() 패턴 사용
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    # next()를 사용하기 위해 이터레이터를 만듭니다.
    print(f"🔍 {SAMPLE_COUNT}개의 샘플을 가져옵니다...")
    sample_data_iterator = iter(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset)인 경우
    print(f"🔍 {SAMPLE_COUNT}개의 샘플을 가져옵니다...")
    # List로 변환하는 것이 가장 안전합니다.
    sample_data_list = list(dataset.take(SAMPLE_COUNT))
    sample_data_iterator = iter(sample_data_list)

# 4. 데이터 분석 (창의적이고 신나는 실습!)
print("\n" + "=" * 80)
print("🧬 Step 4: '조건별 분석'로 AI를 속여보자! (메타데이터 탐색)")
print("=" * 80)

print("\n💡 [실습 목표]: 이 지역의 이미지가 구름이나 눈에 가려졌을 때, 분류 결과(labels)가 어떻게 달라질지 가설을 세워봅시다.")

# 샘플 데이터를 수동으로 리스트에 저장하며 분석을 진행합니다.
for i in range(SAMPLE_COUNT):
    try:
        # 반복문 안에서 next()를 사용하여 샘플을 안전하게 조회합니다.
        sample = next(sample_data_iterator)
        print(f"\n--- [샘플 {i+1} 분석] ---")
        
        # 필수 메타데이터 추출 (Key-Value 쌍으로 이해하기 쉽게)
        is_cloud = sample['features']['contains_cloud_or_shadow']
        has_snow = sample['features']['contains_seasonal_snow']
        labels = sample['features']['labels']
        
        print(f"📸 날씨 조건: 구름/그림자={is_cloud}, 계절 눈={has_snow}")
        print(f"🏷️ 분류 레이블 (labels): {', '.join(labels)}")
        
        # ----------------------------------------------------------------------
        # 🎯 AI 튜터의 해석: 
        # 1. 'contains_cloud_or_shadow'를 확인하세요. (불리언 값)
        # 2. 만약 True라면? -> 우리의 목표(Classification)를 방해하는 가장 큰 장애물입니다!
        # 3. 만약 True라면, labels가 비어 있거나 매우 불분명할 가능성이 높습니다. (모델이 혼란스러움)
        # ----------------------------------------------------------------------
        
        if is_cloud:
            print("   => 🧐 코멘트: 구름이 가려진 사진입니다. 이 경우, AI는 '어디에 무엇이 있는지'를 정확히 알기 어렵습니다. 분류 정확도가 떨어질 수 있어요!")
        elif has_snow and labels:
            print("   => ✨ 코멘트: 눈이 있지만 레이블이 명확합니다. 이 데이터셋은 '눈이 있는 환경'에 대한 분류도 학습했네요!")
        else:
            print("   => 👍 코멘트: 날씨 조건이 비교적 좋아 보이는 샘플입니다. AI가 학습하기 가장 좋은 '황금 샘플'이죠!")

    except StopIteration:
        print("\n✨ 모든 샘플에 대한 분석이 완료되었습니다!")
        break


# 5. 데이터 분석 요약 및 결론 (가장 중요한 AI 학습의 결론)
print("\n" + "=" * 80)
print("🏆 최종 결론: 데이터 기반의 멋진 AI 엔지니어링!")
print("=" * 80)

print("""
✅ 핵심 배운 것 요약:
1.  **데이터 특성 이해:** 이 데이터는 단순히 이미지만 주지 않고, '구름 여부', '눈 여부' 같은 부가적인 조건(Metadata)을 함께 제공합니다.
2.  **실습한 AI 기술:**
    *   **Conditioning:** "만약 구름이 없다(False)는 조건 하에서만, 도시 지역(label='urban')의 분류 레이블을 확인하자." 와 같은 질문을 던지는 것이 고차원적인 AI 활용입니다.
    *   **결과 해석:** 샘플을 직접 보면서 "오, 구름 때문에 이 분류 결과는 믿기 어렵겠군"이라고 판단하는 것이 바로 AI 모델을 사용하는 최고의 능력입니다!
""")